<a href="https://colab.research.google.com/github/UAMCAntwerpen/2040FBDBIC/blob/main/11_Docking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install rdkit-pypi

     |████████████████████████████████| 19.7 MB 1.2 MB/s 


In [4]:
!pip install py3Dmol

In [5]:
import rdkit
from rdkit import Chem
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import Draw
IPythonConsole.ipython_useSVG=True 
import py3Dmol
from rdkit.Chem import AllChem
import copy
from rdkit.Chem.rdMolAlign import AlignMol

In [6]:
# The molecule from the crystal structure
mol = Chem.MolFromSmiles('C[NH+]1CCN(CC1)C(=O)Nc1ccc(F)cc1')
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol)
AllChem.MMFFOptimizeMolecule(mol)

mblock = Chem.MolToMolBlock(mol)
viewer = py3Dmol.view(width=500, height=500)
viewer.addModel(mblock, 'mol')
viewer.setStyle({"stick":{}})
viewer.zoomTo()

You appear to be running in JupyterLab (or JavaScript failed to load for some other reason). You need to install the 3dmol extension: 
 jupyter labextension install jupyterlab_3dmol

In [40]:
# Get and prepare the ligand database
import requests
url = "https://raw.githubusercontent.com/UAMCAntwerpen/2040FBDBIC/main/compounds_10k.smi"
smiles = requests.get(url).text.split("\n")

inputMols = []
for s in smiles:
  if s != "":
    mol = Chem.MolFromSmiles(s)
    if mol is not None:
      mol.SetProp("_Name", s)
      inputMols.append(mol)

print(len(inputMols))

# Keep only the first 10 molecules for reasons of demo
inputMols = inputMols[:10]
print(len(inputMols))

10000
10


In [41]:
def generateconformations(mol, nconfs, name):
    mol = Chem.AddHs(mol)
    ids = AllChem.EmbedMultipleConfs(mol, nconfs)
    for id in ids:
        AllChem.UFFOptimizeMolecule(mol, confId=id)
    return mol, list(ids), name

In [42]:
import multiprocessing

# Download this from http://pypi.python.org/pypi/futures
from concurrent import futures
import progressbar

ConfoutputFilePath = 'InputForDocking.sdf'
writer = Chem.SDWriter(ConfoutputFilePath)

numcores = multiprocessing.cpu_count()
max_workers = numcores
with futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
  jobs = []
  for mol in inputMols:
    if mol:
      name = mol.GetProp('_Name')
      job = executor.submit(generateconformations, mol, 10, name)
      jobs.append(job)

    widgets = ["Generating conformations; ", progressbar.Percentage(), " ",
               progressbar.ETA(), " ", progressbar.Bar()]
    pbar = progressbar.ProgressBar(widgets=widgets, maxval=len(jobs))
    for job in pbar(futures.as_completed(jobs)):
      mol, ids, name = job.result()
      mol.SetProp('_Name', name)
      for id in ids:
        writer.write(mol, confId=id)
writer.close()

Generating conformations; 100% Time:  0:00:00 |###############################|
Generating conformations; 100% Time:  0:00:00 |###############################|
Generating conformations; 100% Time:  0:00:00 |###############################|
Generating conformations; 100% Time:  0:00:01 |###############################|
Generating conformations; 100% Time:  0:00:00 |###############################|
Generating conformations; 100% Time:  0:00:00 |###############################|
Generating conformations; 100% Time:  0:00:00 |###############################|
Generating conformations; 100% Time:  0:00:00 |###############################|
Generating conformations; 100% Time:  0:00:01 |###############################|
Generating conformations; 100% Time:  0:00:00 |###############################|


In [43]:
!wget https://sourceforge.net/projects/smina/files/smina.static/download -O smina.static

--2021-08-27 12:45:45--  https://sourceforge.net/projects/smina/files/smina.static/download
Resolving sourceforge.net (sourceforge.net)... 204.68.111.105
Connecting to sourceforge.net (sourceforge.net)|204.68.111.105|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://downloads.sourceforge.net/project/smina/smina.static?ts=gAAAAABhKN55QyWegXGNcSPrc_nJuW6uOox6e0Ih5sgGJAi8-19grFXGy3MiLMAIkTXSXLthFzoGIUoJ5z45aBjn7PsVN2cXiA%3D%3D&use_mirror=managedway&r= [following]
--2021-08-27 12:45:45--  https://downloads.sourceforge.net/project/smina/smina.static?ts=gAAAAABhKN55QyWegXGNcSPrc_nJuW6uOox6e0Ih5sgGJAi8-19grFXGy3MiLMAIkTXSXLthFzoGIUoJ5z45aBjn7PsVN2cXiA%3D%3D&use_mirror=managedway&r=
Resolving downloads.sourceforge.net (downloads.sourceforge.net)... 204.68.111.105
Connecting to downloads.sourceforge.net (downloads.sourceforge.net)|204.68.111.105|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://managedway.dl.sourceforge.n

In [44]:
!chmod u+x smina.static

In [45]:
!./smina.static --help


Input:
  -r [ --receptor ] arg         rigid part of the receptor (PDBQT)
  --flex arg                    flexible side chains, if any (PDBQT)
  -l [ --ligand ] arg           ligand(s)
  --flexres arg                 flexible side chains specified by comma 
                                separated list of chain:resid or 
                                chain:resid:icode
  --flexdist_ligand arg         Ligand to use for flexdist
  --flexdist arg                set all side chains within specified distance 
                                to flexdist_ligand to flexible

Search space (required):
  --center_x arg                X coordinate of the center
  --center_y arg                Y coordinate of the center
  --center_z arg                Z coordinate of the center
  --size_x arg                  size in the X dimension (Angstroms)
  --size_y arg                  size in the Y dimension (Angstroms)
  --size_z arg                  size in the Z dimension (Angstroms)
  --autobox_ligan

In [48]:
# Get the protein files
!wget https://raw.githubusercontent.com/UAMCAntwerpen/2040FBDBIC/main/protein_minus_ligand.pdb
!wget https://raw.githubusercontent.com/UAMCAntwerpen/2040FBDBIC/main/373ligand_only.pdb
!wget https://raw.githubusercontent.com/UAMCAntwerpen/2040FBDBIC/main/protein_plus_373ligand.pdb
DockedFilePath = 'All_Docked.sdf'
#FlexibleDockedFilePath = 'FlexDocked.sdf'

--2021-08-27 13:02:22--  https://raw.githubusercontent.com/UAMCAntwerpen/2040FBDBIC/main/protein_minus_ligand.pdb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 670849 (655K) [text/plain]
Saving to: ‘protein_minus_ligand.pdb.3’

protein_minus_ligan 100%[===================>] 655.13K  --.-KB/s    in 0.05s   

2021-08-27 13:02:22 (13.8 MB/s) - ‘protein_minus_ligand.pdb.3’ saved [670849/670849]

--2021-08-27 13:02:22--  https://raw.githubusercontent.com/UAMCAntwerpen/2040FBDBIC/main/373ligand_only.pdb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response..

In [47]:
!./smina.static --cpu 8 --seed 0 --autobox_ligand 373ligand_only.pdb -r protein_minus_ligand.pdb -l '{ConfoutputFilePath}' -o '{DockedFilePath}'

   _______  _______ _________ _        _______ 
  (  ____ \(       )\__   __/( (    /|(  ___  )
  | (    \/| () () |   ) (   |  \  ( || (   ) |
  | (_____ | || || |   | |   |   \ | || (___) |
  (_____  )| |(_)| |   | |   | (\ \) ||  ___  |
        ) || |   | |   | |   | | \   || (   ) |
  /\____) || )   ( |___) (___| )  \  || )   ( |
  \_______)|/     \|\_______/|/    )_)|/     \|


smina is based off AutoDock Vina. Please cite appropriately.

Weights      Terms
-0.035579    gauss(o=0,_w=0.5,_c=8)
-0.005156    gauss(o=3,_w=2,_c=8)
0.840245     repulsion(o=0,_c=8)
-0.035069    hydrophobic(g=0.5,_b=1.5,_c=8)
-0.587439    non_dir_h_bond(g=-0.7,_b=0,_c=8)
1.923        num_tors_div

Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
1       -

In [49]:
def DrawDocking(protein, ligand):
  complex_pl = Chem.MolToPDBBlock(protein)
  docked_pdb = Chem.MolToPDBBlock(ligand)
  viewer = py3Dmol.view(width=800,height=800)
  viewer.addModel(complex_pl,'pdb')
  viewer.addModel(docked_pdb)
  prot = {'resn': ["DMS", "UNL", "SO4", "LIG", "HOH", "Cl"], 'invert': 1}  #define prot as all except list
  viewer.setStyle(prot,{'cartoon': {'colorscheme':'ssPyMol'}}) # Colour by secondary structure
  Lig_373 = {'resn' : 'LIG'} #original ligand in pdb file
  MyLig = {'resn':'UNL'} #ligand to be added from docking
  viewer.addSurface(py3Dmol.VDW,{'opacity':0.7, 'color': 'white'}, prot)
  viewer.setStyle({'resi': '132'}, {'stick': {'colorscheme': 'blueCarbon'}})
  viewer.setStyle({'resi': '147'}, {'stick': {'colorscheme': 'blueCarbon'}})
  viewer.setStyle({'resi': '311'}, {'stick': {'colorscheme': 'blueCarbon'}})
  viewer.setStyle(Lig_373,{'stick':{'colorscheme': 'whiteCarbon','radius':.1}}) 
  viewer.setStyle(MyLig,{'stick':{'colorscheme' : 'greenCarbon'}})
  viewer.zoomTo(MyLig)
  return viewer  

In [50]:
mols = [m for m in Chem.SDMolSupplier(DockedFilePath)]

OSError: ignored

In [53]:
receptor = Chem.MolFromPDBFile("protein_plus_373ligand.pdb")
DrawDocking(receptor,mols[0])

NameError: ignored